# Piedra Papel Tijera con Q-Learning
Implementación de un agente que aprende a jugar usando reinforcement learning.

In [19]:
import numpy as np
import random

In [20]:
JUGADAS = {0: 'Piedra', 1: 'Papel', 2: 'Tijera'}

## Entorno del juego
El entorno maneja las reglas y devuelve recompensa según el resultado.

In [21]:
class PiedraPapelTijeraEnv:
    def __init__(self):
        self.reset()
        
    def reset(self):
        self.last_opponent_move = None
        return self.last_opponent_move
    
    def step(self, agent_move, opponent_move):
        if agent_move == opponent_move:
            reward = 0
            result = 'Empate'
        elif (agent_move == 0 and opponent_move == 2) or \
             (agent_move == 1 and opponent_move == 0) or \
             (agent_move == 2 and opponent_move == 1):
            reward = 1
            result = 'Victoria Agente'
        else:
            reward = -1
            result = 'Victoria Oponente'
            
        next_state = opponent_move
        self.last_opponent_move = opponent_move
        return next_state, reward, result

## Agente 
El agente usa Q-learning

In [22]:
class Agent:
    def __init__(self, alpha=0.05, c=2.0):
        self.alpha = alpha
        self.c = c
        self.Q = {}
        self.N = {}
        self.N_s = {}
        
    def _inicializar_estado(self, state):
        if state not in self.Q:
            self.Q[state] = np.zeros(3)
            self.N[state] = np.zeros(3)
            self.N_s[state] = 0
            
    def move(self, state, explore=True):
        self._inicializar_estado(state)
        
        if not explore:
            return np.argmax(self.Q[state])
        
        if self.N_s[state] == 0:
            return random.choice([0, 1, 2])
        
        acciones_no_probadas = [a for a in range(3) if self.N[state][a] == 0]
        if acciones_no_probadas:
            return random.choice(acciones_no_probadas)
        
        total_visitas_s = self.N_s[state]
        valores_seleccion = np.zeros(3)
        for a in range(3):
            termino_exploracion = self.c * np.sqrt(np.log(total_visitas_s) / self.N[state][a])
            valores_seleccion[a] = self.Q[state][a] + termino_exploracion
            
        max_val = np.max(valores_seleccion)
        mejores_acciones = np.where(valores_seleccion == max_val)[0]
        return random.choice(mejores_acciones)
    
    def update(self, state, action, reward):
        self._inicializar_estado(state)
        self.N_s[state] += 1
        self.N[state][action] += 1
        self.Q[state][action] += self.alpha * (reward - self.Q[state][action])

## Oponente con patrón fijo
Este oponente siempre juega en ciclo: Piedra → Papel → Tijera → ...

In [23]:
class OponentePatron:
    def __init__(self):
        self.last_move = None
        
    def move(self, state):
        if self.last_move is None:
            self.last_move = random.choice([0, 1, 2])
        else:
            self.last_move = (self.last_move + 1) % 3
        return self.last_move

## Simulación de entrenamiento
Se entrena el agente durante N rondas contra el oponente patrón.

In [24]:
def simular_juego(rondas=500):
    env = PiedraPapelTijeraEnv()
    agente = Agent(alpha=0.1, c=1.0)
    rival = OponentePatron()
    
    aciertos_acumulados = 0
    estado = env.reset()
    
    print("--- INICIANDO SIMULACIÓN DE ENTRENAMIENTO ---")
    for r in range(1, rondas + 1):
        jugada_rival = rival.move(estado)
        jugada_agente = agente.move(estado, explore=True)
        nuevo_estado, recompensa, resultado = env.step(jugada_agente, jugada_rival)
        agente.update(estado, jugada_agente, recompensa)
        
        if recompensa == 1:
            aciertos_acumulados += 1
        estado = nuevo_estado
        
        if r % 50 == 0:
            tasa = aciertos_acumulados / r
            print(f"Ronda {r:03d} | Tasa de victorias: {tasa:.2%}")
            
    print("\n--- TABLA Q APRENDIDA ---")
    for s in [None, 0, 1, 2]:
        nom_estado = 'Inicio' if s is None else JUGADAS[s]
        valores = [round(val, 3) for val in agente.Q.get(s, np.zeros(3))]
        print(f"Estado [{nom_estado}]: Piedra={valores[0]}, Papel={valores[1]}, Tijera={valores[2]}")

In [25]:
simular_juego(rondas=500)

--- INICIANDO SIMULACIÓN DE ENTRENAMIENTO ---
Ronda 050 | Tasa de victorias: 74.00%
Ronda 100 | Tasa de victorias: 84.00%
Ronda 150 | Tasa de victorias: 89.33%
Ronda 200 | Tasa de victorias: 92.00%
Ronda 250 | Tasa de victorias: 92.40%
Ronda 300 | Tasa de victorias: 92.67%
Ronda 350 | Tasa de victorias: 93.71%
Ronda 400 | Tasa de victorias: 94.50%
Ronda 450 | Tasa de victorias: 95.11%
Ronda 500 | Tasa de victorias: 95.60%

--- TABLA Q APRENDIDA ---
Estado [Inicio]: Piedra=0.0, Papel=0.0, Tijera=0.0
Estado [Piedra]: Piedra=-0.271, Papel=0.0, Tijera=1.0
Estado [Papel]: Piedra=1.0, Papel=-0.271, Tijera=0.0
Estado [Tijera]: Piedra=0.0, Papel=1.0, Tijera=-0.271


## Jugar contra el agente
Función para jugar manualmente contra la IA.

In [26]:
def jugar_contra_humano(rondas=10):
    env = PiedraPapelTijeraEnv()
    agente = Agent(alpha=0.3, c=0.8)
    
    print(" ¡Bienvenido a Piedra, Papel o Tijera contra la IA! ")
    print(" El agente aprenderá de tus jugadas anteriores.    ")
    print("Instrucciones: Escribe 0 para Piedra, 1 para Papel, 2 para Tijera. Escribe 'salir' para terminar.\n")
    
    estado = env.reset()
    marcador_agente = 0
    marcador_humano = 0
    empates = 0
    
    for r in range(1, rondas + 1):
        print(f"--- Ronda {r} ---")
        entrada = input("Elige tu jugada (0: Piedra, 1: Papel, 2: Tijera): ").strip().lower()
        if entrada == 'salir':
            print("Juego terminado por el usuario.")
            break
            
        if entrada not in ['0', '1', '2']:
            print("Entrada no válida. Escribe 0, 1 o 2.")
            continue
            
        jugada_humano = int(entrada)
        jugada_agente = agente.move(estado, explore=True)
        nuevo_estado, recompensa, resultado = env.step(jugada_agente, jugada_humano)
        agente.update(estado, jugada_agente, recompensa)
        
        print(f"Tú elegiste: {JUGADAS[jugada_humano]}")
        print(f"El agente eligió: {JUGADAS[jugada_agente]}")
        print(f"Resultado: {resultado}")
        
        if recompensa == 1:
            marcador_agente += 1
        elif recompensa == -1:
            marcador_humano += 1
        else:
            empates += 1
            
        print(f"Marcador actual -> Tú: {marcador_humano} | Agente: {marcador_agente} | Empates: {empates}\n")
        estado = nuevo_estado
        
    print("              FIN DEL JUEGO                ")
    print(f"Marcador Final -> Tú: {marcador_humano} | Agente: {marcador_agente} | Empates: {empates}")

In [27]:
jugar_contra_humano(rondas=15)

 ¡Bienvenido a Piedra, Papel o Tijera contra la IA! 
 El agente aprenderá de tus jugadas anteriores.    
Instrucciones: Escribe 0 para Piedra, 1 para Papel, 2 para Tijera. Escribe 'salir' para terminar.

--- Ronda 1 ---
Tú elegiste: Papel
El agente eligió: Piedra
Resultado: Victoria Oponente
Marcador actual -> Tú: 1 | Agente: 0 | Empates: 0

--- Ronda 2 ---
Tú elegiste: Piedra
El agente eligió: Papel
Resultado: Victoria Agente
Marcador actual -> Tú: 1 | Agente: 1 | Empates: 0

--- Ronda 3 ---
Tú elegiste: Piedra
El agente eligió: Piedra
Resultado: Empate
Marcador actual -> Tú: 1 | Agente: 1 | Empates: 1

--- Ronda 4 ---
Tú elegiste: Papel
El agente eligió: Tijera
Resultado: Victoria Agente
Marcador actual -> Tú: 1 | Agente: 2 | Empates: 1

--- Ronda 5 ---
Tú elegiste: Tijera
El agente eligió: Tijera
Resultado: Empate
Marcador actual -> Tú: 1 | Agente: 2 | Empates: 2

--- Ronda 6 ---
Tú elegiste: Papel
El agente eligió: Papel
Resultado: Empate
Marcador actual -> Tú: 1 | Agente: 2 | Empa